# Construct connectivity and functional matrices for the 906 cohort

Built on the canonical pipeline from `structural_network.ipynb`:
1. **Cohort** — 906 neurons that have BOTH a clean proofread axon (`ax_clean`) AND a coregistered functional recording (`tuning='matched'`). Spans V1+RL+AL across 13 different scans.
2. **Connectivity matrix** `C` — `(N, N)` directed, weighted by total synaptic `size` from neuron *i* to neuron *j*. Built from `data/1718/raw/synapses_matched.csv`. Autapses removed.
3. **Functional matrix** `F` — `(N, M)` mean firing rate per neuron per stimulus, where columns are aligned by `condition_hash`. Because the 906 neurons live in different scans (each scan shows its own trial sequence), we use the intersection of condition hashes seen in *every* involved session — so every neuron has a response for every column.

Row alignment: `C[i, :]` and `F[i, :]` describe the same neuron, identified by `matched.iloc[i]`.

In [1]:
from pathlib import Path

import h5py
import numpy as np
import pandas as pd
from scipy import sparse

import microns_datacleaner as mic
import microns_datacleaner.filters as fl

In [2]:
# Build the canonical 906-neuron cohort exactly the way structural_network.ipynb does.
# proofread='ax_clean' -> 2,192 neurons with reliably reconstructed outgoing axons.
# tuning='matched'     -> intersect with neurons that have a coregistered functional recording.
DATA_DIR = Path('..') / 'data'
SYN_CSV  = DATA_DIR / '1718' / 'raw' / 'synapses_matched.csv'
FUNC_H5  = DATA_DIR / 'functional' / 'microns_functional.h5'

cleaner = mic.MicronsDataCleaner(datadir='../data/', version=1718, download_policy='minimum')
units, _ = cleaner.process_nucleus_data(functional_data='best_only')
matched = (
    fl.filter_neurons(fl.filter_neurons(units, proofread='ax_clean'), tuning='matched')
    .reset_index(drop=True)
)
matched.index.name = 'matrix_idx'   # row/col index in C and F

print(f'cohort size: {len(matched)}')
print(matched.brain_area.value_counts().to_dict())
print(matched.layer.value_counts().to_dict())
matched.head()


Transform positions:   0%|          | 0/94014 [00:00<?, ?it/s]


Transform positions:  18%|█▊        | 17147/94014 [00:00<00:00, 171463.21it/s]


Transform positions:  37%|███▋      | 34419/94014 [00:00<00:00, 172196.31it/s]


Transform positions:  55%|█████▍    | 51681/94014 [00:00<00:00, 172385.15it/s]


Transform positions:  73%|███████▎  | 68920/94014 [00:00<00:00, 172142.47it/s]


Transform positions:  92%|█████████▏| 86135/94014 [00:00<00:00, 171946.89it/s]


Transform positions: 100%|██████████| 94014/94014 [00:00<00:00, 171928.93it/s]

cohort size: 906
{'V1': 728, 'RL': 122, 'AL': 56}
{'L2/3': 348, 'L4': 348, 'L5': 156, 'L6': 53, 'L1': 1}


,nucleus_id,pt_root_id,pt_position_x,pt_position_y,pt_position_z,classification_system,cell_type,brain_area,strategy_axon,strategy_dendrite,session,scan_idx,unit_id,pref_ori,pref_dir,gOSI,gDSI,cc_abs,tuning_type,layer
matrix_idx,,,,,,,,,,,,,,,,,,,,
0,223016,864691135763941174,613.394783,104.264055,927.88,excitatory_neuron,23P,V1,axon_partially_extended,dendrite_extended,6.0,4.0,2112.0,123.26255,303.26254,0.476260,0.071378,0.480216,matched,L2/3
1,222257,864691136488446226,547.738750,131.413065,711.88,excitatory_neuron,23P,V1,axon_partially_extended,dendrite_extended,6.0,2.0,2156.0,99.99531,279.99530,0.505451,0.070855,0.412397,matched,L2/3
2,260263,864691135694415551,620.053865,292.504745,810.64,excitatory_neuron,4P,V1,axon_partially_extended,dendrite_clean,9.0,3.0,2087.0,177.07890,177.07890,0.449638,0.287918,0.654134,matched,L4
3,291326,864691135398817569,705.839136,101.366084,901.68,excitatory_neuron,23P,V1,axon_fully_extended,dendrite_extended,6.0,4.0,1576.0,79.81989,79.81989,0.228730,0.065927,0.361165,matched,L2/3
4,256465,864691134942472035,669.207813,164.654284,834.56,excitatory_neuron,23P,V1,axon_fully_extended,dendrite_extended,9.0,4.0,7995.0,156.53249,336.53250,0.096377,0.038825,0.659531,matched,L2/3


## 1. Connectivity matrix `C`

`C[i, j]` = total synaptic `size` (cleft volume) from neuron *i* (presynaptic) onto neuron *j* (postsynaptic), with autapses removed. The synapse table was downloaded by `cleaner.download_synapse_data(matched_ids, matched_ids)` and merged into `synapses_matched.csv` — so it already contains only V1↔V1 (in the broader cohort sense, including RL+AL) connections.

In [3]:
pt_to_idx = pd.Series(matched.index.values, index=matched.pt_root_id)
N = len(matched)

syn = pd.read_csv(SYN_CSV)
syn = syn[syn.pre_pt_root_id != syn.post_pt_root_id]                                          # drop autapses
syn = syn[syn.pre_pt_root_id.isin(pt_to_idx.index) & syn.post_pt_root_id.isin(pt_to_idx.index)]

rows = pt_to_idx.loc[syn.pre_pt_root_id].to_numpy()
cols = pt_to_idx.loc[syn.post_pt_root_id].to_numpy()
data = syn['size'].to_numpy(dtype=np.float32)

# COO with duplicate (row, col) entries -> CSR collapses them by summing.
C = sparse.coo_matrix((data, (rows, cols)), shape=(N, N)).tocsr()
print(f'C: shape={C.shape}, nnz={C.nnz}, density={C.nnz / (N * N):.2e}')

C: shape=(906, 906), nnz=11822, density=1.44e-02


## 2. Functional matrix `F`

The 906 neurons are spread across ~13 different `(session, scan_idx)` scans. Each scan presents its own trial sequence, so trial *k* in scan A is generally a different stimulus from trial *k* in scan B. To make a coherent F, we **align columns by `condition_hash`** (the stimulus identifier attached to each trial in the h5).

We use the **intersection** of condition hashes seen by *every* involved session — the ~116 "oracle" stimuli — so every cohort neuron has a response for every column. For each neuron and condition, we average over all trials of that condition in its native session, and over time within each trial → one scalar firing rate per (neuron, stimulus).

In [4]:
cohort = matched.assign(
    session_key=matched.session.astype(int).astype(str) + '_' + matched.scan_idx.astype(int).astype(str),
    unit_id_int=matched.unit_id.astype(int),
)
session_keys = sorted(cohort.session_key.unique())

with h5py.File(FUNC_H5, 'r') as f:
    # 1. trial -> condition_hash, per session
    trials_cache = {}
    cond_sets = []
    for sk in session_keys:
        trials = f[f'sessions/{sk}/trials']
        ch = [(tk, trials[tk].attrs['condition_hash']) for tk in trials.keys()]
        trials_cache[sk] = ch
        cond_sets.append({h for _, h in ch})

    common = sorted(set.intersection(*cond_sets))                  # oracle conditions
    cond_to_col = {h: j for j, h in enumerate(common)}
    M = len(common)
    print(f'sessions involved: {len(session_keys)}, oracle conditions: {M}')

    # 2. fill F session-by-session
    F = np.full((N, M), np.nan, dtype=np.float32)
    for sk in session_keys:
        sess = cohort[cohort.session_key == sk]
        h5_uids = f[f'sessions/{sk}/meta/unit_ids'][:]
        uid_to_row = {int(u): i for i, u in enumerate(h5_uids)}

        matrix_idxs, h5_rows = [], []
        for mi, uid in zip(sess.index, sess.unit_id_int):
            r = uid_to_row.get(uid)
            if r is not None:
                matrix_idxs.append(mi)
                h5_rows.append(r)
        if not h5_rows:
            continue
        matrix_idxs = np.array(matrix_idxs)
        h5_rows = np.array(h5_rows)

        # group trials by condition, restricted to oracle conditions
        per_cond_trial_means = {}    # cond_hash -> list of (n_cohort_in_sess,) arrays
        trials_grp = f[f'sessions/{sk}/trials']
        for tk, ch in trials_cache[sk]:
            if ch not in cond_to_col:
                continue
            resp = trials_grp[tk]['responses'][:]                  # (n_units_in_sess, T)
            per_cond_trial_means.setdefault(ch, []).append(resp[h5_rows].mean(axis=1))

        for ch, tms in per_cond_trial_means.items():
            cond_mean = np.mean(np.stack(tms, axis=0), axis=0)     # avg across trials of this condition
            F[matrix_idxs, cond_to_col[ch]] = cond_mean

print(f'F: shape={F.shape}, NaN frac={np.isnan(F).mean():.3f}, mean={np.nanmean(F):.3f}, std={np.nanstd(F):.3f}')

sessions involved: 13, oracle conditions: 116


F: shape=(906, 116), NaN frac=0.000, mean=1.896, std=3.200


## 3. Sanity check

Row *i* of both `C` and `F` refers to the same neuron — `matched.iloc[i]` carries `pt_root_id`, `session`, `scan_idx`, `unit_id`, brain area, layer, and cell type.

In [5]:
assert C.shape[0] == F.shape[0] == len(matched)
out_deg = np.asarray(C.sum(axis=1)).ravel()
in_deg  = np.asarray(C.sum(axis=0)).ravel()
print(f'N neurons        : {len(matched)}')
print(f'C nonzero edges  : {C.nnz}')
print(f'mean out-strength: {out_deg.mean():.1f}   mean in-strength: {in_deg.mean():.1f}')
print(f'F shape          : {F.shape}    (cols = oracle stimulus conditions)')
print(f'F NaN fraction   : {np.isnan(F).mean():.3f}')
matched.head()

N neurons        : 906
C nonzero edges  : 11822
mean out-strength: 121413.0   mean in-strength: 121413.0
F shape          : (906, 116)    (cols = oracle stimulus conditions)
F NaN fraction   : 0.000


,nucleus_id,pt_root_id,pt_position_x,pt_position_y,pt_position_z,classification_system,cell_type,brain_area,strategy_axon,strategy_dendrite,session,scan_idx,unit_id,pref_ori,pref_dir,gOSI,gDSI,cc_abs,tuning_type,layer
matrix_idx,,,,,,,,,,,,,,,,,,,,
0,223016,864691135763941174,613.394783,104.264055,927.88,excitatory_neuron,23P,V1,axon_partially_extended,dendrite_extended,6.0,4.0,2112.0,123.26255,303.26254,0.476260,0.071378,0.480216,matched,L2/3
1,222257,864691136488446226,547.738750,131.413065,711.88,excitatory_neuron,23P,V1,axon_partially_extended,dendrite_extended,6.0,2.0,2156.0,99.99531,279.99530,0.505451,0.070855,0.412397,matched,L2/3
2,260263,864691135694415551,620.053865,292.504745,810.64,excitatory_neuron,4P,V1,axon_partially_extended,dendrite_clean,9.0,3.0,2087.0,177.07890,177.07890,0.449638,0.287918,0.654134,matched,L4
3,291326,864691135398817569,705.839136,101.366084,901.68,excitatory_neuron,23P,V1,axon_fully_extended,dendrite_extended,6.0,4.0,1576.0,79.81989,79.81989,0.228730,0.065927,0.361165,matched,L2/3
4,256465,864691134942472035,669.207813,164.654284,834.56,excitatory_neuron,23P,V1,axon_fully_extended,dendrite_extended,9.0,4.0,7995.0,156.53249,336.53250,0.096377,0.038825,0.659531,matched,L2/3


In [6]:
# Peek at C and F as DataFrames.
# C is sparse (906x906, mostly zeros) — the dense top-left corner is uninformative,
# so the most useful view is the edge list: (pre, post, size) for the first edges.
print('--- C as edge list ---')
C_coo = C.tocoo()
C_df  = pd.DataFrame({'pre_matrix_idx': C_coo.row, 'post_matrix_idx': C_coo.col, 'size': C_coo.data})
display(C_df.head())

print('\n--- C dense slice (rows 0..4, cols 0..9, mostly 0) ---')
display(pd.DataFrame(C[:5, :10].toarray()))

print('\n--- F as DataFrame (rows = neurons, first 10 oracle stimulus columns) ---')
display(pd.DataFrame(F).iloc[:, :10].head())

--- C as edge list ---


,pre_matrix_idx,post_matrix_idx,size
0,0,157,5044.0
1,0,191,11860.0
2,0,419,9852.0
3,0,422,9400.0
4,0,434,11884.0



--- C dense slice (rows 0..4, cols 0..9, mostly 0) ---


,0,1,2,3,4,5,6,7,8,9
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1712.0,0.0,0.0



--- F as DataFrame (rows = neurons, first 10 oracle stimulus columns) ---


,0,1,2,3,4,5,6,7,8,9
0,1.649365,1.182109,0.226775,0.592331,3.598964e-08,0.603698,4.085823,1.151602,0.782649,0.733773
1,3.459434,1.750683,0.943060,0.805471,2.806359e+00,9.528574,3.632434,1.728102,0.931030,0.177283
2,0.605065,0.485904,0.702298,0.101977,1.763449e+00,3.326710,0.263543,0.400099,1.042906,2.248633
3,3.778404,0.969280,0.102051,1.565679,1.894320e-01,2.369151,0.581383,0.166733,1.316849,0.887342
4,4.520851,0.608446,0.171833,1.196882,2.352277e+00,2.996230,0.549466,2.349010,0.275764,3.420527


## 4. Functional similarity matrix `F_corr`

`F` itself isn't a network — it's a `(neurons × stimuli)` activity table. To turn it into a *functional network* we compute pairwise similarity between rows. Two natural choices:

- **Pearson correlation** — centers each neuron's response vector (subtracts its mean across stimuli) before measuring similarity. Captures whether two neurons go up and down *together* across stimuli, ignoring their absolute firing rates. This is the standard choice in MICrONS / V1 structure-function work.
- **Cosine similarity** — same as Pearson but *without* centering. Captures angular alignment of the raw response vectors. Two neurons that are both highly active on the same stimuli look similar even if neither has much variation. Use this if you want to weight overall response magnitude as part of "similarity."

The two agree when neurons have similar mean rates; they diverge for cells with very different baselines. We compute both so you can pick.

In [7]:
# Functional similarity = pairwise comparison of each neuron's stimulus-response vector.
# Pearson via np.corrcoef (centers + scales). Cosine via L2-normalized dot product.
F_corr_pearson = np.corrcoef(F)                                       # (906, 906)

F_unit = F / np.linalg.norm(F, axis=1, keepdims=True)                 # row-normalize
F_corr_cosine  = F_unit @ F_unit.T                                    # (906, 906)

# Pick which one to use for downstream analysis. Pearson is the typical default.
F_corr = F_corr_pearson

print(f'F_corr_pearson: shape={F_corr_pearson.shape}  diag mean={np.diag(F_corr_pearson).mean():.3f}  off-diag mean={F_corr_pearson[~np.eye(N,dtype=bool)].mean():.3f}')
print(f'F_corr_cosine : shape={F_corr_cosine.shape}   diag mean={np.diag(F_corr_cosine).mean():.3f}  off-diag mean={F_corr_cosine[~np.eye(N,dtype=bool)].mean():.3f}')

# Structure -> function check: connected pairs should be more functionally similar.
A_undir   = (C + C.T).astype(bool).toarray()                          # symmetric: connected either way
np.fill_diagonal(A_undir, False)
off_diag  = ~np.eye(N, dtype=bool)
connected = A_undir & off_diag
unconn    = (~A_undir) & off_diag

print(f'\n--- structure vs function (using Pearson F_corr) ---')
print(f'connected pairs  : n={connected.sum():>7,d}   mean F_corr={F_corr[connected].mean():+.4f}')
print(f'unconnected pairs: n={unconn.sum():>7,d}   mean F_corr={F_corr[unconn].mean():+.4f}')
print(f'difference       : {F_corr[connected].mean() - F_corr[unconn].mean():+.4f}')

F_corr_pearson: shape=(906, 906)  diag mean=1.000  off-diag mean=0.023
F_corr_cosine : shape=(906, 906)   diag mean=1.000  off-diag mean=0.468

--- structure vs function (using Pearson F_corr) ---
connected pairs  : n= 23,160   mean F_corr=+0.0557
unconnected pairs: n=796,770   mean F_corr=+0.0223
difference       : +0.0334


## 5. Visualizations

Two views of what we just built:

1. **Correlation heatmap** — `F_corr` as a 906×906 image with the viridis colormap. The diagonal reads 1 (a neuron is perfectly correlated with itself); the off-diagonal mass is the structure to look at.
2. **3D structural network** — neurons placed at their actual `(pt_position_x, pt_position_y, pt_position_z)` cortical coordinates, edges drawn between connected pairs (from `C`) and colored by their functional correlation. Black background, fully interactive (drag to rotate, scroll to zoom, hover for neuron metadata). Built with `plotly` so it renders natively in VS Code's Jupyter without the third-party-content prompt.

In [ ]:
# 5.1  Correlation matrix heatmap (viridis)
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(F_corr, cmap='viridis', aspect='equal', vmin=-1, vmax=1)
ax.set_title(f'Functional similarity F_corr  ({F_corr.shape[0]} x {F_corr.shape[1]})')
ax.set_xlabel('neuron j')
ax.set_ylabel('neuron i')
fig.colorbar(im, ax=ax, label='Pearson correlation', shrink=0.85)
plt.tight_layout()
plt.show()

In [22]:
# 5.2  Interactive 3D structural network — uses plotly so it renders natively in VS Code Jupyter.
# Drag to rotate, scroll to zoom, hover a neuron for metadata.
# Click a brain-area name in the legend to toggle that region's neurons + bounding box.
# Double-click an area to isolate it (hide the others).
import plotly.graph_objects as go
from matplotlib.colors import Normalize, to_hex
from matplotlib import cm

xyz = matched[['pt_position_x', 'pt_position_y', 'pt_position_z']].to_numpy()

# --- edges from C, colored by F_corr (binned for per-trace coloring) ---
A_und = (C + C.T).tocoo()
mask  = A_und.row < A_und.col
edges = np.stack([A_und.row[mask], A_und.col[mask]], axis=1)
edge_corr = F_corr[edges[:, 0], edges[:, 1]]
print(f'edges drawn: {len(edges):,}   corr range: [{edge_corr.min():+.3f}, {edge_corr.max():+.3f}]')

n_bins = 16
vlim   = max(abs(np.percentile(edge_corr, 1)), abs(np.percentile(edge_corr, 99)))
bins   = np.linspace(-vlim, +vlim, n_bins + 1)
cmap   = cm.coolwarm
norm   = Normalize(-vlim, +vlim)

traces = []
for b in range(n_bins):
    lo, hi = bins[b], bins[b + 1]
    sel = (edge_corr >= lo) & (edge_corr <= hi if b == n_bins - 1 else edge_corr < hi)
    if not sel.any():
        continue
    e   = edges[sel]
    pts = np.empty((len(e) * 3, 3))
    pts[0::3] = xyz[e[:, 0]]
    pts[1::3] = xyz[e[:, 1]]
    pts[2::3] = np.nan
    mid     = 0.5 * (lo + hi)
    color   = to_hex(cmap(norm(mid)))
    opacity = float(np.clip(abs(mid) / vlim, 0.1, 0.8))
    traces.append(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode='lines',
        line=dict(color=color, width=1),
        opacity=opacity, hoverinfo='skip', showlegend=False,
        legendgroup='edges',
    ))

# --- per-brain-area: translucent bounding box + colored neuron dots in same legend group ---
AREA_COLORS = {'V1': '#90EE90', 'RL': '#FFFF99', 'AL': '#87CEFA'}      # light green / yellow / blue

def cube_mesh(pts):
    """8 vertices and 12 triangular faces of an axis-aligned bounding box."""
    mn, mx = pts.min(axis=0), pts.max(axis=0)
    X = [mn[0], mx[0], mx[0], mn[0], mn[0], mx[0], mx[0], mn[0]]
    Y = [mn[1], mn[1], mx[1], mx[1], mn[1], mn[1], mx[1], mx[1]]
    Z = [mn[2], mn[2], mn[2], mn[2], mx[2], mx[2], mx[2], mx[2]]
    i = [0, 0, 0, 0, 4, 4, 6, 6, 4, 0, 3, 2]
    j = [1, 2, 4, 1, 5, 6, 5, 2, 0, 1, 6, 3]
    k = [2, 3, 5, 5, 6, 7, 1, 1, 7, 4, 7, 6]
    return X, Y, Z, i, j, k

for area in ['V1', 'RL', 'AL']:
    sel = (matched.brain_area == area).to_numpy()
    if not sel.any():
        continue
    pts   = xyz[sel]
    color = AREA_COLORS[area]
    grp   = f'area_{area}'

    X, Y, Z, ii, jj, kk = cube_mesh(pts)
    traces.append(go.Mesh3d(
        x=X, y=Y, z=Z, i=ii, j=jj, k=kk,
        color=color, opacity=0.08, flatshading=True,
        hoverinfo='skip', showlegend=False, legendgroup=grp,
    ))
    traces.append(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode='markers',
        marker=dict(size=2.5, color=color, line=dict(width=0)),
        name=f'{area}  ({sel.sum()})',
        text=[f'idx={i}<br>{area} {l}<br>{ct}'
              for i, l, ct in zip(matched.index[sel], matched.layer[sel], matched.cell_type[sel])],
        hoverinfo='text', legendgroup=grp, showlegend=True,
    ))

# --- invisible trace just to draw the F_corr colorbar ---
traces.append(go.Scatter3d(
    x=[xyz[0, 0]], y=[xyz[0, 1]], z=[xyz[0, 2]], mode='markers',
    marker=dict(
        size=0.001, color=[0], cmin=-vlim, cmax=+vlim, colorscale='RdBu_r',
        colorbar=dict(title=dict(text='F_corr', font=dict(color='white')),
                      tickfont=dict(color='white'), x=1.02, len=0.7),
    ),
    hoverinfo='skip', showlegend=False,
))

fig = go.Figure(data=traces)
fig.update_layout(
    paper_bgcolor='black', plot_bgcolor='black',
    font=dict(color='white'),
    title=dict(text='Structural network in 3D — neurons colored by brain area, edges by F_corr',
               font=dict(color='white')),
    legend=dict(
        font=dict(color='white'),
        bgcolor='rgba(0,0,0,0.5)', bordercolor='gray', borderwidth=1,
        itemclick='toggle', itemdoubleclick='toggleothers',
        x=0.01, y=0.99,
    ),
    scene=dict(
        bgcolor='black', aspectmode='data',
        xaxis=dict(title='x (µm)', backgroundcolor='black', color='white', gridcolor='dimgray'),
        yaxis=dict(title='y (µm)', backgroundcolor='black', color='white', gridcolor='dimgray'),
        zaxis=dict(title='z (µm)', backgroundcolor='black', color='white', gridcolor='dimgray'),
    ),
    width=1000, height=800, margin=dict(l=0, r=0, b=0, t=40),
)
fig.show()

edges drawn: 11,580   corr range: [-0.382, +0.892]
